# Notebook 9 — Cross-Border Interconnections

Italy is connected to 5 neighbouring electricity markets. This notebook explores how cross-border exchanges work in the simulator.

Topics:
1. Price areas — stochastic foreign market prices (O-U, Cholesky-correlated)
2. Import as virtual generator — enters the merit order
3. Export as post-dispatch adjustment
4. Reliability models — stochastic cable faults
5. Economic and CO2 benefits of interconnections

**Runtime**: ~60 seconds

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from energy_sim.config import (
    ITALIAN_MIX, GAS_SCENARIOS, P_PEAK_GW,
    QUARTERS_PER_DAY, PRICE_AREAS, PRICE_AREA_CORRELATIONS,
    INTERCONNECTIONS, STORAGE_UNITS,
)
from energy_sim.simulation import run_monte_carlo
from energy_sim.price_areas import PriceAreaCoupling, build_price_areas_from_config
from energy_sim.models import TimeGrid

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

## 1. Price Areas: Italy's Neighbours

Each neighbouring market is modelled as an independent O-U process, with optional **Cholesky-correlated** shocks (European markets co-move because of shared weather and gas prices).

In [ ]:
print(f"{'Area':<5s} {'mu':>6s} {'sigma':>6s} {'theta':>6s} {'CI (gCO2/kWh)':>15s}")
print("-" * 45)
for area, params in PRICE_AREAS.items():
    print(f"{area:<5s} {params['mu']:6.0f} {params['sigma']:6.0f} "
          f"{params['theta']:6.2f} {params['carbon_intensity_g_per_kwh']:15.0f}")

In [ ]:
# Generate correlated price paths
tg = TimeGrid()
price_areas = build_price_areas_from_config(PRICE_AREAS)
coupling = PriceAreaCoupling(price_areas, PRICE_AREA_CORRELATIONS, correlated=True)

rng = np.random.default_rng(42)
paths = coupling.generate_paths(tg.n, rng)

days = np.arange(tg.n) / QUARTERS_PER_DAY

fig, ax = plt.subplots(figsize=(14, 5))
colors = {'FR': 'blue', 'CH': 'red', 'AT': 'green', 'SI': 'orange', 'GR': 'purple'}
for area, path in paths.items():
    ax.plot(days, path, lw=0.3, color=colors[area], alpha=0.7, label=f'{area} (mu={PRICE_AREAS[area]["mu"]:.0f})')
ax.set_xlabel('Day of year')
ax.set_ylabel('Price (EUR/MWh)')
ax.set_title('Correlated foreign market prices (one MC realization)')
ax.legend()
plt.tight_layout()
plt.show()

# Show empirical correlations
area_names = list(paths.keys())
corr_matrix = np.corrcoef([paths[a] for a in area_names])
print("\nEmpirical correlation matrix (price levels):")
print(f"{'':>5s}", '  '.join(f'{a:>5s}' for a in area_names))
for i, a in enumerate(area_names):
    print(f"{a:>5s}", '  '.join(f'{corr_matrix[i,j]:5.2f}' for j in range(len(area_names))))

## 2. Interconnection Topology

Each link has import/export NTC, transport cost, and a reliability model:

In [ ]:
print(f"{'Link':<8s} {'Area':>5s} {'Import (GW)':>12s} {'Export (GW)':>12s} "
      f"{'Transport':>10s} {'Reliability':>15s}")
print("-" * 70)
for name, cfg in INTERCONNECTIONS.items():
    rel_type = cfg['reliability']['type']
    rel_tech = cfg['reliability'].get('tech', '-')
    print(f"{name:<8s} {cfg['price_area']:>5s} {cfg['ntc_import_gw']:12.1f} "
          f"{cfg['ntc_export_gw']:12.1f} {cfg['transport_cost_eur_mwh']:10.1f} "
          f"{rel_tech:>15s}")

total_import = sum(c['ntc_import_gw'] for c in INTERCONNECTIONS.values())
total_export = sum(c['ntc_export_gw'] for c in INTERCONNECTIONS.values())
print(f"\nTotal NTC: {total_import:.1f} GW import, {total_export:.1f} GW export")

## 3. Running with interconnections

Let's compare the simulation with and without cross-border exchanges:

In [ ]:
# Without interconnections
mc_no_ic = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
)

# With interconnections
mc_ic = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
    interconnections_cfg=INTERCONNECTIONS,
    price_areas_cfg=PRICE_AREAS,
    price_area_correlations=PRICE_AREA_CORRELATIONS,
)

print(f"{'Metric':<30s} {'No IC':>10s} {'With IC':>10s} {'Delta':>10s}")
print("-" * 65)
for metric, key, fmt in [
    ('Avg price (EUR/MWh)', 'avg_price', '.1f'),
    ('Carbon intensity (gCO2/kWh)', 'carbon_intensity', '.0f'),
    ('Emissions (Mt CO2)', 'total_emissions', '.2f'),
    ('Avg inertia (s)', 'avg_inertia', '.2f'),
]:
    v_no = mc_no_ic[key].mean()
    v_ic = mc_ic[key].mean()
    if key == 'total_emissions':
        v_no /= 1e6; v_ic /= 1e6
    delta = v_ic - v_no
    print(f"{metric:<30s} {v_no:>10{fmt}} {v_ic:>10{fmt}} {delta:>+10{fmt}}")

In [ ]:
# Net imports per link
ic_names = mc_ic['interconnection_names']
net_twh = mc_ic['net_import_twh'].mean(axis=0)  # avg across runs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Net imports bar chart
bar_colors = ['green' if v > 0 else 'red' for v in net_twh]
axes[0].barh(ic_names, net_twh, color=bar_colors)
axes[0].set_xlabel('Net import (TWh, positive = import)')
axes[0].set_title('Average annual net imports by link')
axes[0].axvline(0, color='black', lw=0.5)

# Economic benefit
econ_benefit = mc_ic['total_economic_benefit_eur'].mean(axis=0) / 1e6  # M EUR
axes[1].barh(ic_names, econ_benefit, color='steelblue')
axes[1].set_xlabel('Economic benefit (M EUR/year)')
axes[1].set_title('Congestion-rent benefit by link')

plt.tight_layout()
plt.show()

# CO2 benefit
co2_benefit = mc_ic['total_co2_benefit_tons'].mean(axis=0) / 1e3  # ktCO2
print("\nCO2 benefit per link (ktCO2/year, positive = avoided emissions):")
for name, val in zip(ic_names, co2_benefit):
    print(f"  {name}: {val:+.0f} ktCO2")

## 4. Play with parameters

Toggle faults on/off to see the impact of reliability:

In [ ]:
# ── PLAY WITH THESE ──────────────────────────────────────
enable_faults = False  # try True vs False
# ─────────────────────────────────────────────────────────

mc_test = run_monte_carlo(
    ITALIAN_MIX, GAS_SCENARIOS['base'],
    n_runs=10, seed=42,
    interconnections_cfg=INTERCONNECTIONS,
    price_areas_cfg=PRICE_AREAS,
    price_area_correlations=PRICE_AREA_CORRELATIONS,
    enable_ntc_faults=enable_faults,
)

print(f"With faults={'ON' if enable_faults else 'OFF'}:")
print(f"  Avg price: {mc_test['avg_price'].mean():.1f} EUR/MWh")
print(f"  Net import (total): {mc_test['net_import_twh'].mean(axis=0).sum():.2f} TWh")

## Key Takeaways

1. **Imports enter the merit order** as virtual generators with `SRMC = foreign_price + transport_cost`. When cheaper than domestic gas, the system imports.
2. **Exports** happen when domestic prices are below the foreign market minus transport cost — additional generation is dispatched to fill export NTC.
3. **Cholesky correlation** ensures neighbouring markets co-move realistically (FR-CH highly correlated, FR-GR weakly).
4. **Reliability models** (TwoStateMarkov) create stochastic faults — especially relevant for submarine cables (IT-GR) with long repair times.
5. The **economic benefit** (congestion rent) and **CO2 benefit** quantify the value of each link. France (nuclear-heavy, low CI) provides the most value.

**Next notebook**: [10 — Battery Storage](./10_battery_storage.ipynb) — arbitrage and capacity sizing.